## Tools
Models can request to call tools that perform tasks such as fetching data from a db, searching the web, or running code. Tools are pairings of :

1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ["GROQ_API_KEY"]= os.getenv("groq")

model= init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the human sense of forming ideas and expressing them with language. What they do is **vocal mimicry**—they copy sounds they hear, especially those that are frequent, salient, and socially rewarding. Several biological and environmental factors make this ability especially pronounced in parrots:\n\n| Factor | How it contributes to “talking” |\n|--------|--------------------------------|\n| **Highly developed vocal apparatus** | Parrots have a flexible syrinx (the bird equivalent of a voice box) and a muscular tongue that can produce a wide range of frequencies and timbres, from high‑pitched whistles to human‑like consonants. |\n| **Large brain regions for sound learning** | The **nidopallium caudomediale (NCM)** and **caudomedial nidopallium (NCM)** in the avian brain are analogous to parts of the human auditory cortex. These areas are enlarged in parrots, giving them excellent auditory memory and the ability to compare heard sounds with stored

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str)-> str:
    """Get the weather at a location"""
    return f"It's Sunny in {location}"

In [4]:
model_with_tools=model.bind_tools([get_weather])


In [6]:
response=model_with_tools.invoke("what is the weather right now in gurugram")
print(response)
for tool_call in response.tool_calls:
    print(tool_call)
    print(f"Tool: {tool_call['name']}")
    print(f"ARGS: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks for current weather in Gurugram. We need to call get_weather function with location "Gurugram".', 'tool_calls': [{'id': 'fc_d604d7f3-bc9d-48e8-a698-6afd1ae2fe2c', 'function': {'arguments': '{"location":"Gurugram"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 130, 'total_tokens': 187, 'completion_time': 0.122972937, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.004908864, 'prompt_tokens_details': None, 'queue_time': 0.052310705, 'total_time': 0.127881801}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9be0-d28a-7081-8af0-c85640ccf91b-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Gurugram'}, 'id': 'fc_d604d7f3-bc9d-48e8-a698-6afd1ae2fe2c', 'type': 'tool_cal

### Tool execution loops

In [9]:
messages = [{'role': "user","content":"Whats the weather in Boston?"}]
ai_msg =  model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result)
print(messages)
final_response = model_with_tools.invoke(messages)
print(final_response.content)


[{'role': 'user', 'content': 'Whats the weather in Boston?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_b068db92-3347-486c-8ee1-e117ec7fbc02', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 126, 'total_tokens': 169, 'completion_time': 0.091159734, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.004981475, 'prompt_tokens_details': None, 'queue_time': 0.047125675, 'total_time': 0.096141209}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e9be2-2838-7162-9fbd-2536a21e2501-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_b068db92-3347-486c-8e